<a href="https://colab.research.google.com/github/aayushcs007/AI-LAB-1BF24CS007-/blob/main/1BF24CS007_Week5_PropositionalLogic_TruthTableEnumeration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd
import itertools

def evaluate(expr, model):
    return eval(expr, {}, model)

def generate_interactive_truth_table(symbols, kb, queries):
    rows = []
    num_symbols = len(symbols)

    for truth_values in itertools.product([True, False], repeat=num_symbols):
        model = dict(zip(symbols, truth_values))

        kb_evals = [evaluate(clause, model) for clause in kb]
        kb_true = all(kb_evals)

        # Evaluate each query in the list dynamically
        query_evals = {f"Query: {q}": evaluate(q, model) for q in queries}

        row = {sym: model[sym] for sym in symbols}
        row.update({'KB': kb_true})
        row.update(query_evals)
        rows.append(row)

    df = pd.DataFrame(rows)

    # Check entailment status for each query individually
    kb_true_rows = df[df['KB'] == True]
    entailment_results = {}

    for q in queries:
        col_name = f"Query: {q}"
        if kb_true_rows.empty:
            entailment_results[q] = True
        else:
            entailment_results[q] = kb_true_rows[col_name].all()

    return df, entailment_results

if __name__ == "__main__":
    print("=== Propositional Logic Multi-Query Verification Tool ===")
    print("Note: Use standard Python syntax operators: 'and', 'or', 'not'\n")

    raw_symbols = input("Enter all proposition symbols separated by commas (e.g., p, q, r): ")
    symbols = [sym.strip() for sym in raw_symbols.split(",") if sym.strip()]

    print("\nEnter your Knowledge Base (KB) clauses one by one.")
    print("Type 'done' when you are finished entering rules.")
    kb_clauses = []
    while True:
        clause = input(f"Enter KB Clause #{len(kb_clauses) + 1}: ").strip()
        if clause.lower() == 'done':
            if not kb_clauses:
                print("Error: The KB cannot be completely empty.")
                continue
            break
        if clause:
            kb_clauses.append(clause)

    # New Multi-Query Input Loop
    print("\nEnter your Query expressions one by one.")
    print("Type 'done' when you are finished entering queries.")
    queries = []
    while True:
        q = input(f"Enter Query #{len(queries) + 1}: ").strip()
        if q.lower() == 'done':
            if not queries:
                print("Error: You must enter at least one query.")
                continue
            break
        if q:
            queries.append(q)

    print("\n[Processing Execution Matrix...]")
    df_result, results = generate_interactive_truth_table(symbols, kb_clauses, queries)

    print("\n=== GENERATED TRUTH TABLE ===")
    print(df_result.to_string(index=False))
    print("==============================")

    print("\n=== FINAL ENTAILMENT RESULTS ===")
    for q, does_entail in results.items():
        print(f"Does KB entail '{q}'? -> **{'YES' if does_entail else 'NO'}**")
    print("=================================\n")


=== Propositional Logic Multi-Query Verification Tool ===
Note: Use standard Python syntax operators: 'and', 'or', 'not'

Enter all proposition symbols separated by commas (e.g., p, q, r): p,q,r

Enter your Knowledge Base (KB) clauses one by one.
Type 'done' when you are finished entering rules.
Enter KB Clause #1: not q or p
Enter KB Clause #2: not p or (not q)
Enter KB Clause #3: q or r
Enter KB Clause #4: done

Enter your Query expressions one by one.
Type 'done' when you are finished entering queries.
Enter Query #1: r
Enter Query #2: not r or p
Enter Query #3: not q or r
Enter Query #4: done

[Processing Execution Matrix...]

=== GENERATED TRUTH TABLE ===
    p     q     r    KB  Query: r  Query: not r or p  Query: not q or r
 True  True  True False      True               True               True
 True  True False False     False               True              False
 True False  True  True      True               True               True
 True False False False     False          